In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data_clean/clean_masterdata.csv')
df = df.drop_duplicates(subset=['userId', 'movieId']).reset_index(drop=True)


df['user_cat'] = df['userId'].astype('category')
df['movie_cat'] = df['title'].astype('category')


unique_movies = df[['title', 'movie_cat', 'overview', 'genres']].drop_duplicates(subset=['movie_cat'])
unique_movies = unique_movies.sort_values(by='movie_cat').reset_index(drop=True)


unique_movies['mix'] = unique_movies['overview'].fillna('') + " " + unique_movies['genres'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_movies['mix']) 


sparse_matrix = csr_matrix((df['rating'], (df['movie_cat'].cat.codes, df['user_cat'].cat.codes)))
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(sparse_matrix) 


unique_movies['row_num'] = unique_movies['movie_cat'].cat.codes
indices = pd.Series(unique_movies['row_num'].values, index=unique_movies['title'].str.lower().str.strip())
indices = indices[~indices.index.duplicated(keep='first')]



In [4]:
def get_hybrid_recommendation(title, content_weight=0.5, collab_weight=0.5):
    clean_title = title.lower().strip()
    
    if clean_title not in indices:
        return f"'{title}' is not in the database."
        
    idx = indices[clean_title]
    
    # 2A Score:
    content_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    #  2B Score:
    collab_scores = cosine_similarity(latent_matrix[idx].reshape(1, -1), latent_matrix).flatten()
    
    hybrid_scores = (content_scores * content_weight) + (collab_scores * collab_weight)
    
    sim_scores = list(enumerate(hybrid_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    top_10 = sim_scores[1:11]
    movie_indices = [i[0] for i in top_10]
    reverse_mapping = movie_mapping.set_index('row_num')['title']
    
    recommendations = reverse_mapping.loc[movie_indices].tolist()
    for i, movie in enumerate(recommendations, 1):
        print(f"{i}. {movie}")

get_hybrid_recommendation('The Dark Knight', content_weight=0.5, collab_weight=0.5)

1. The Dark Knight Rises
2. Batman Begins
3. Inception
4. Inglourious Basterds
5. The Departed
6. Iron Man
7. The Prestige
8. V for Vendetta
9. Star Trek
10. Avatar
